# 04 - Batch Download Gol.gg Game Pages

Download and archive 84 Gol.gg game pages from the 2026
Worlds Main Event, covering game IDs 83721 through 83638.

Raw HTML and API responses will be saved locally for
subsequent parsing and database creation.

In [1]:
import os
import re
import json
import time
import requests
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv
from IPython.display import display

In [4]:
#locate project folders
current_dir = Path.cwd()

project_root = None

for folder in [current_dir, *current_dir.parents]:
    if (folder / "data" / "raw" / "html").exists():
        project_root = folder
        break

if project_root is None:
    raise FileNotFoundError("Could not locate project data folder")

# Define directories
html_dir = project_root / "data" / "raw" / "html"
json_dir = project_root / "data" / "raw" / "json"
processed_dir = project_root / "data" / "processed"

# Create directories if they don't exist
html_dir.mkdir(parents=True, exist_ok=True)
json_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("HTML directory:", html_dir)
print("JSON directory:", json_dir)

Project root: c:\Users\boyle\League_Of_Legends_Herald_Analysis
HTML directory: c:\Users\boyle\League_Of_Legends_Herald_Analysis\data\raw\html
JSON directory: c:\Users\boyle\League_Of_Legends_Herald_Analysis\data\raw\json


In [5]:
#generate our 84 game ids
START_GAME_ID = 83721
END_GAME_ID = 83638

game_ids = list(range(START_GAME_ID, END_GAME_ID - 1, -1))

print("Number of games:", len(game_ids))
print("First game:", game_ids[0])
print("Last game:", game_ids[-1])

Number of games: 84
First game: 83721
Last game: 83638


In [6]:
#load api credentials

load_dotenv(project_root / "golgg_scraper" / ".env")

STEADY_API_TOKEN = os.getenv("STEADY_API_TOKEN")

if not STEADY_API_TOKEN:
    raise ValueError("STEADY_API_TOKEN not found in .env")

STEADY_API_URL = "https://api.steadyapi.com/v1/scrapeflow/scraper"

headers = {
    "Authorization": f"Bearer {STEADY_API_TOKEN}",
    "Content-Type": "application/json"
}

print("SteadyAPI token loaded successfully.")

SteadyAPI token loaded successfully.


In [9]:
# SteadyAPI configuration

STEADY_API_BASE_URL = os.getenv(
    "STEADY_API_BASE_URL",
    "https://api.steadyapi.com"
)

SCRAPEFLOW_ENDPOINT = (
    f"{STEADY_API_BASE_URL}/v1/scrapeflow/scraper"
)

request_headers = {
    "Authorization": f"Bearer {STEADY_API_TOKEN}",
    "Content-Type": "application/json",
    "Accept": "application/json",
}

print("ScrapeFlow endpoint:", SCRAPEFLOW_ENDPOINT)
print("API headers configured successfully.")

ScrapeFlow endpoint: https://api.steadyapi.com/v1/scrapeflow/scraper
API headers configured successfully.


In [10]:
#download function

def download_game(game_id, max_retries=3):
    """
    Download one Gol.gg game page through SteadyAPI.

    Saves:
        data/raw/html/game_{game_id}.html
        data/raw/json/game_{game_id}.json

    Returns a dictionary describing the result.
    """

    game_id = str(game_id)

    # Construct the target URL and output filenames.
    game_url = f"https://gol.gg/game/stats/{game_id}/page-game/"

    html_path = html_dir / f"game_{game_id}.html"
    json_path = json_dir / f"game_{game_id}.json"

    # Skip games that have already been downloaded successfully.
    if (
        html_path.exists()
        and html_path.stat().st_size > 0
        and json_path.exists()
        and json_path.stat().st_size > 0
    ):
        return {
            "game_id": game_id,
            "status": "skipped",
            "message": "Files already exist"
        }

    # Use the same request configuration as Notebook 01.
    request_payload = {
        "url": game_url,
        "method": "GET",
        "geo": "US",
        "render_js": False,
        "timeout": 20,
        "output": "html",
    }

    for attempt in range(1, max_retries + 1):

        try:
            response = requests.post(
                SCRAPEFLOW_ENDPOINT,
                headers=request_headers,
                json=request_payload,
                timeout=45,
            )

            # Check the API request itself.
            response.raise_for_status()

            response_data = response.json()

            # Check the target webpage response.
            meta = response_data.get("meta", {})
            target_status = meta.get("status")

            if target_status != 200:
                raise ValueError(
                    f"Target page returned status: {target_status}"
                )

            # Extract the HTML returned by SteadyAPI.
            html_content = response_data.get("body")

            if not isinstance(html_content, str) or not html_content.strip():
                raise ValueError(
                    "Response body is missing, empty, or not an HTML string"
                )

            # Save the original HTML.
            html_path.write_text(
                html_content,
                encoding="utf-8"
            )

            # Save the complete API response as JSON.
            with open(json_path, "w", encoding="utf-8") as f:
                json.dump(response_data, f, indent=2, ensure_ascii=False)

            return {
                "game_id": game_id,
                "status": "success",
                "message": f"Downloaded successfully ({len(html_content)} characters)"
            }

        except Exception as e:

            error_message = str(e)

            # Retry failed requests.
            if attempt < max_retries:
                wait_time = 2 * attempt
                print(
                    f"Game {game_id}: attempt {attempt} failed. "
                    f"Retrying in {wait_time} seconds..."
                )
                time.sleep(wait_time)

            else:
                return {
                    "game_id": game_id,
                    "status": "failed",
                    "message": error_message
                }


In [11]:
#test for one game

test_result = download_game(83721)

print(test_result)

{'game_id': '83721', 'status': 'success', 'message': 'Downloaded successfully (626352 characters)'}


In [12]:
# Check that both files exist and have reasonable sizes.

html_path = html_dir / "game_83721.html"
json_path = json_dir / "game_83721.json"

print("HTML exists:", html_path.exists())
print("HTML size:", html_path.stat().st_size, "bytes")

print("\nJSON exists:", json_path.exists())
print("JSON size:", json_path.stat().st_size, "bytes")

HTML exists: True
HTML size: 627293 bytes

JSON exists: True
JSON size: 630007 bytes


In [ ]:
# Load and inspect the downloaded HTML.

from bs4 import BeautifulSoup

html = html_path.read_text(encoding="utf-8")
soup = BeautifulSoup(html, "lxml")

print("Page title:", soup.title.get_text(strip=True) if soup.title else "Not found")

print("Blue team header:", bool(soup.find("div", class_="blue-line-header")))
print("Red team header:", bool(soup.find("div", class_="red-line-header")))
print("Timeline labels:", bool(soup.find(class_="title_blue")) and bool(soup.find(class_="title_red")))

Page title: BLG vs T1 game 4 - Worlds 2026 Main Event SEMIFINALS - Games of Legends
Blue team header: True
Red team header: True
Timeline labels: True


In [14]:
#download all 84 games

download_results = []

for index, game_id in enumerate(game_ids, start=1):

    print(f"[{index}/{len(game_ids)}] Downloading game {game_id}...")

    result = download_game(game_id)

    download_results.append(result)

    print(f"    Status: {result['status']}")

    # Pause between requests.
    time.sleep(1.2)


# Create a DataFrame containing all download results.
download_df = pd.DataFrame(download_results)

print("\nDownload summary:")
display(download_df["status"].value_counts().to_frame("count"))

print("\nFailed downloads:")
display(download_df[download_df["status"] == "failed"])

[1/84] Downloading game 83721...
    Status: skipped
[2/84] Downloading game 83720...
    Status: success
[3/84] Downloading game 83719...
    Status: success
[4/84] Downloading game 83718...
    Status: success
[5/84] Downloading game 83717...
Game 83717: attempt 1 failed. Retrying in 2 seconds...
Game 83717: attempt 2 failed. Retrying in 4 seconds...
    Status: failed
[6/84] Downloading game 83716...
Game 83716: attempt 1 failed. Retrying in 2 seconds...
Game 83716: attempt 2 failed. Retrying in 4 seconds...
    Status: failed
[7/84] Downloading game 83715...
Game 83715: attempt 1 failed. Retrying in 2 seconds...
Game 83715: attempt 2 failed. Retrying in 4 seconds...
    Status: failed
[8/84] Downloading game 83714...
Game 83714: attempt 1 failed. Retrying in 2 seconds...
Game 83714: attempt 2 failed. Retrying in 4 seconds...
    Status: failed
[9/84] Downloading game 83713...
Game 83713: attempt 1 failed. Retrying in 2 seconds...
Game 83713: attempt 2 failed. Retrying in 4 seconds.

KeyboardInterrupt: 

In [15]:
# Review the downloads completed before stopping.

partial_df = pd.DataFrame(download_results)

print("Games attempted:", len(partial_df))

print("\nStatus summary:")
display(partial_df["status"].value_counts().to_frame("count"))

print("\nFailed games:")
display(partial_df[partial_df["status"] == "failed"])

Games attempted: 22

Status summary:


,count
status,
failed,18
success,3
skipped,1



Failed games:


,game_id,status,message
4,83717,failed,403 Client Error: Forbidden for url: https://a...
5,83716,failed,403 Client Error: Forbidden for url: https://a...
6,83715,failed,403 Client Error: Forbidden for url: https://a...
7,83714,failed,403 Client Error: Forbidden for url: https://a...
8,83713,failed,403 Client Error: Forbidden for url: https://a...
9,83712,failed,403 Client Error: Forbidden for url: https://a...
10,83711,failed,403 Client Error: Forbidden for url: https://a...
11,83710,failed,403 Client Error: Forbidden for url: https://a...
12,83709,failed,403 Client Error: Forbidden for url: https://a...
13,83708,failed,403 Client Error: Forbidden for url: https://a...


In [16]:
game_id = "83717"

game_url = f"https://gol.gg/game/stats/{game_id}/page-game/"

request_payload = {
    "url": game_url,
    "method": "GET",
    "geo": "US",
    "render_js": False,
    "timeout": 20,
    "output": "html",
}

response = requests.post(
    SCRAPEFLOW_ENDPOINT,
    headers=request_headers,
    json=request_payload,
    timeout=45,
)

print("API HTTP status:", response.status_code)

response_data = response.json()

meta = response_data.get("meta", {})

print("Target page status:", meta.get("status"))
print("Target page status_code:", meta.get("status_code"))
print("Response message:", response_data.get("message"))
print("Response keys:", list(response_data.keys()))

body = response_data.get("body")

print(
    "Body type:", type(body).__name__,
    "| Body length:", len(body) if isinstance(body, str) else "N/A"
)

API HTTP status: 403
Target page status: None
Target page status_code: None
Response message: Your free plan includes 20 requests per day, and you have used them for today. Upgrade to a paid plan for a full monthly limit: https://steadyapi.com/user/subscriptions/pricing
Response keys: ['success', 'message']
Body type: NoneType | Body length: N/A


In [18]:
#our API has a strict limit

# lets see if we can directly scrape off the website 


import requests

# Check Gol.gg's published crawler instructions first.
robots_url = "https://gol.gg/robots.txt"

robots_response = requests.get(
    robots_url,
    timeout=20
)

print("Robots.txt HTTP status:", robots_response.status_code)
print(robots_response.text[:5000])


Robots.txt HTTP status: 200
User-agent: *
Disallow: /adm_lms/
Disallow: /meta/
Disallow: /week/
Disallow: /raw/
Disallow: /game/ajax.score.php
Disallow: /league-admin/
Disallow: /user/
Disallow: /summoner/
Disallow: /ranking/

User-agent: DataForSeoBot
Disallow: /

User-agent: MJ12bot
Disallow: /

User-agent: ias_crawler
Disallow: /

User-agent: SemrushBot
Disallow: /

User-agent: TTD-Content
Disallow: /

User-agent: CriteoBot/0.1
Disallow: /

User-agent: dotbot
Disallow: /

User-agent: AwarioSmartBot
Disallow: /

User-agent: BLEXBot
Disallow: /

User-agent: AhrefsBot
Disallow: /

User-agent: Amazonbot
Disallow: /

User-agent: meta-externalagent
Disallow: /

User-agent: MojeekBot
Crawl-delay: 30

User-agent: Yandex
Crawl-delay: 30

User-agent: PetalBot
Crawl-delay: 30

User-agent: coccocbot-web
Crawl-delay: 30

User-agent: ShapBot
Disallow: /


In [19]:
#looks like scraping the game pages directly is allowed

#lets now try direct parsing


import requests
from pathlib import Path

game_id = "83721"

game_url = f"https://gol.gg/game/stats/{game_id}/page-game/"

headers = {
    "User-Agent": "Mozilla/5.0 (compatible; LeagueDataResearch/1.0)"
}

response = requests.get(
    game_url,
    headers=headers,
    timeout=30
)

print("HTTP status:", response.status_code)
print("HTML characters:", len(response.text))
print("Content type:", response.headers.get("Content-Type"))

response.raise_for_status()

# Save the HTML using our existing folder structure.
html_path = html_dir / f"game_{game_id}.html"

html_path.write_text(
    response.text,
    encoding="utf-8"
)

print("Saved:", html_path)


HTTP status: 200
HTML characters: 626352
Content type: text/html; charset=UTF-8
Saved: c:\Users\boyle\League_Of_Legends_Herald_Analysis\data\raw\html\game_83721.html


In [21]:
#does the parser still work?

def parse_game_page(html, game_id):
    """
    Parse a Gol.gg game page and extract:
    1. Blue and red team names
    2. Winning team and winning side
    3. Timeline team abbreviations
    4. Timeline events, timestamps, and event ownership

    Returns:
        game_info: Dictionary containing game metadata
        events: List of dictionaries containing timeline events
    """

    # --------------------------------------------------
    # 1. Parse the HTML
    # --------------------------------------------------

    soup = BeautifulSoup(html, "lxml")

    # --------------------------------------------------
    # 2. Extract team names from the game headers
    # --------------------------------------------------

    blue_header = soup.find("div", class_="blue-line-header")
    red_header = soup.find("div", class_="red-line-header")

    if blue_header is None or red_header is None:
        raise ValueError("Could not find both blue and red team headers")

    blue_link = blue_header.find("a")
    red_link = red_header.find("a")

    if blue_link is None or red_link is None:
        raise ValueError("Could not find team name links")

    blue_team = blue_link.get_text(strip=True)
    red_team = red_link.get_text(strip=True)

    # --------------------------------------------------
    # 3. Identify the winning team
    # --------------------------------------------------

    blue_text = blue_header.get_text(" ", strip=True)
    red_text = red_header.get_text(" ", strip=True)

    blue_won = bool(re.search(r"\bWIN\b", blue_text, re.IGNORECASE))
    red_won = bool(re.search(r"\bWIN\b", red_text, re.IGNORECASE))

    if blue_won and not red_won:
        winning_team = blue_team
        winning_side = "blue"

    elif red_won and not blue_won:
        winning_team = red_team
        winning_side = "red"

    else:
        raise ValueError(
            f"Could not uniquely identify winner: "
            f"blue={blue_text}, red={red_text}"
        )

    # Store the game-level information
    game_info = {
        "game_id": str(game_id),
        "blue_team": blue_team,
        "red_team": red_team,
        "winning_team": winning_team,
        "winning_side": winning_side
    }

    # --------------------------------------------------
    # 4. Find the timeline container
    # --------------------------------------------------

    timeline_container = None

    for div in soup.find_all("div"):

        classes = div.get("class", [])

        if (
            "col-12" in classes
            and "flex-wrap" in classes
            and div.find(class_="title_blue")
            and div.find(class_="title_red")
        ):
            timeline_container = div
            break

    if timeline_container is None:
        raise ValueError("Could not find the Gold graph & Timeline container")

    # --------------------------------------------------
    # 5. Extract timeline team labels
    # --------------------------------------------------

    timeline_blue = timeline_container.find(class_="title_blue")
    timeline_red = timeline_container.find(class_="title_red")

    if timeline_blue is None or timeline_red is None:
        raise ValueError("Could not find both timeline team labels")

    # These are abbreviations, such as GEN and IG.
    # They are not necessarily the same as the full team names.

    timeline_blue_label = timeline_blue.get_text(strip=True)
    timeline_red_label = timeline_red.get_text(strip=True)

    # Add timeline abbreviations to the game metadata.
    game_info["blue_team_timeline_label"] = timeline_blue_label
    game_info["red_team_timeline_label"] = timeline_red_label

    # --------------------------------------------------
    # 6. Map timeline event classes to full team names
    # --------------------------------------------------

    team_names = {
        "blue_action": blue_team,
        "red_action": red_team
    }

    # --------------------------------------------------
    # 7. Extract timeline events
    # --------------------------------------------------

    events = []

    # Get the direct child spans.
    # The first span contains the team labels.
    timeline_spans = timeline_container.find_all(
        "span",
        recursive=False
    )

    # Skip the team-label span.
    event_wrappers = timeline_spans[1:]

    for wrapper in event_wrappers:

        # Each wrapper can contain a blue lane and a red lane.
        lanes = wrapper.find_all(
            "span",
            recursive=False
        )

        for lane in lanes:

            classes = lane.get("class", [])

            # Determine whether this event belongs to blue or red.
            side_class = next(
                (side for side in team_names if side in classes),
                None
            )

            if side_class is None:
                continue

            # Find the event icon.
            img = lane.find("img")

            if img is None:
                continue

            # Extract the timestamp from the lane's text.
            timestamp_match = re.search(
                r"(?<!\d)(\d{1,2}:\d{2})(?!\d)",
                lane.get_text(" ", strip=True)
            )

            if timestamp_match is None:
                continue

            timestamp = timestamp_match.group(1)

            # Convert timestamp into seconds.
            minutes, seconds = map(int, timestamp.split(":"))

            event_time_seconds = minutes * 60 + seconds

            # Determine the side.
            side = (
                "blue"
                if side_class == "blue_action"
                else "red"
            )

            # Save the event.
            events.append({
                "game_id": str(game_id),
                "event_type": img.get("alt"),
                "team": team_names[side_class],
                "side": side,
                "event_time": timestamp,
                "event_time_seconds": event_time_seconds,
                "icon_src": img.get("src")
            })

    # --------------------------------------------------
    # 8. Sort events chronologically
    # --------------------------------------------------

    events.sort(
        key=lambda event: event["event_time_seconds"]
    )

    # Assign event numbers in chronological order.
    for number, event in enumerate(events, start=1):
        event["event_number"] = number

    # --------------------------------------------------
    # 9. Return results
    # --------------------------------------------------

    return game_info, events

html = html_path.read_text(encoding="utf-8")

game_info, events = parse_game_page(html, game_id)

print("Game information:")
print(game_info)

print("\nNumber of timeline events:", len(events))

display(pd.DataFrame(events))

Game information:
{'game_id': '83721', 'blue_team': 'T1', 'red_team': 'Bilibili Gaming', 'winning_team': 'Bilibili Gaming', 'winning_side': 'red', 'blue_team_timeline_label': 'T1', 'red_team_timeline_label': 'BLG'}

Number of timeline events: 11


,game_id,event_type,team,side,event_time,event_time_seconds,icon_src,event_number
0,83721,Hextech Drake,T1,blue,6:31,391,../_img/hextech-dragon.png,1
1,83721,First blood,Bilibili Gaming,red,8:25,505,../_img/firstblood3.png,2
2,83721,Ocean Drake,T1,blue,12:09,729,../_img/ocean-dragon.png,3
3,83721,First Tower,Bilibili Gaming,red,12:42,762,../_img/firsttower2.png,4
4,83721,Rift Herald,Bilibili Gaming,red,15:36,936,../_img/herald-icon.png,5
5,83721,Mountain Drake,Bilibili Gaming,red,17:52,1072,../_img/mountain-dragon.png,6
6,83721,Mountain Drake,Bilibili Gaming,red,23:10,1390,../_img/mountain-dragon.png,7
7,83721,Nashor,T1,blue,29:19,1759,../_img/nashor-icon.png,8
8,83721,Mountain Drake,Bilibili Gaming,red,29:32,1772,../_img/mountain-dragon.png,9
9,83721,Mountain Drake,Bilibili Gaming,red,34:50,2090,../_img/mountain-dragon.png,10


In [22]:
#direct downloader


# Create a reusable HTTP session.
session = requests.Session()

session.headers.update({
    "Accept": "text/html,application/xhtml+xml"
})


def download_game_direct(game_id, max_retries=3):
    """
    Download one Gol.gg game page directly, without an API.

    Saves:
        data/raw/html/game_{game_id}.html

    Returns a dictionary describing the download result.
    """

    game_id = str(game_id)

    game_url = f"https://gol.gg/game/stats/{game_id}/page-game/"

    html_path = html_dir / f"game_{game_id}.html"

    # Skip an existing non-empty HTML file.
    if html_path.exists() and html_path.stat().st_size > 0:
        return {
            "game_id": game_id,
            "status": "skipped",
            "message": "HTML file already exists"
        }

    for attempt in range(1, max_retries + 1):

        try:
            response = session.get(
                game_url,
                timeout=30
            )

            status_code = response.status_code

            # Successful response.
            if status_code == 200:

                html_content = response.text

                if not html_content.strip():
                    return {
                        "game_id": game_id,
                        "status": "failed",
                        "message": "Empty HTML response"
                    }

                # Save the original HTML.
                html_path.write_text(
                    html_content,
                    encoding="utf-8"
                )

                return {
                    "game_id": game_id,
                    "status": "success",
                    "message": f"Downloaded {len(html_content)} characters"
                }

            # These errors are not useful to retry automatically.
            if status_code in [403, 404]:
                return {
                    "game_id": game_id,
                    "status": "failed",
                    "message": f"HTTP {status_code}"
                }

            # Retry rate limits and temporary server errors.
            if status_code == 429 or status_code >= 500:

                if attempt < max_retries:
                    wait_time = 5 * attempt

                    print(
                        f"Game {game_id}: HTTP {status_code}. "
                        f"Retrying in {wait_time} seconds..."
                    )

                    time.sleep(wait_time)
                    continue

                return {
                    "game_id": game_id,
                    "status": "failed",
                    "message": f"HTTP {status_code} after retries"
                }

            # Other unexpected HTTP errors.
            return {
                "game_id": game_id,
                "status": "failed",
                "message": f"Unexpected HTTP status {status_code}"
            }

        except requests.exceptions.RequestException as e:

            if attempt < max_retries:
                wait_time = 5 * attempt

                print(
                    f"Game {game_id}: Request error. "
                    f"Retrying in {wait_time} seconds..."
                )

                time.sleep(wait_time)

            else:
                return {
                    "game_id": game_id,
                    "status": "failed",
                    "message": str(e)
                }


In [23]:
#test one game

test_result = download_game_direct(83720)

print(test_result)

{'game_id': '83720', 'status': 'skipped', 'message': 'HTML file already exists'}


In [24]:
#download all 84 games


# Start a fresh log for the direct-download process.
direct_download_results = []

for index, game_id in enumerate(game_ids, start=1):

    print(f"[{index}/84] Downloading game {game_id}...")

    result = download_game_direct(game_id)

    direct_download_results.append(result)

    print(f"    Status: {result['status']}")
    print(f"    {result['message']}")

    # Save the log after every game.
    direct_download_df = pd.DataFrame(direct_download_results)

    direct_download_df.to_csv(
        processed_dir / "direct_download_log.csv",
        index=False
    )

    # Wait between requests to avoid unnecessary load.
    if index < len(game_ids):
        time.sleep(3)


print("\nFinal download summary:")

display(
    pd.DataFrame(direct_download_results)["status"]
    .value_counts()
    .to_frame("count")
)

print("\nFailed downloads:")

display(
    pd.DataFrame(direct_download_results)
    .query("status == 'failed'")
)


[1/84] Downloading game 83721...
    Status: skipped
    HTML file already exists
[2/84] Downloading game 83720...
    Status: skipped
    HTML file already exists
[3/84] Downloading game 83719...
    Status: skipped
    HTML file already exists
[4/84] Downloading game 83718...
    Status: skipped
    HTML file already exists
[5/84] Downloading game 83717...
    Status: failed
    HTTP 403
[6/84] Downloading game 83716...
    Status: failed
    HTTP 403
[7/84] Downloading game 83715...
    Status: failed
    HTTP 403
[8/84] Downloading game 83714...
    Status: failed
    HTTP 403


KeyboardInterrupt: 